# 🤖 Text Classification — ทำนายหมวดหมู่ มอก. จากขอบข่าย

**Goal:** ใช้ข้อความขอบข่ายภาษาอังกฤษ → ทำนายหมวดหมู่ผลิตภัณฑ์

**Pipeline:** TF-IDF Vectorization → Random Forest / Logistic Regression

**Target:** `tb3_productgroup` (19 หมวดหมู่)

**Evaluation:** Accuracy, Classification Report, Confusion Matrix

In [ ]:
# Google Colab — run this cell first
!pip install pandas scikit-learn plotly openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_excel("std-tisi-b stru.xlsx", skiprows=1)
df.columns = ["เลขที่","ชื่อTH","ประกาศ","ชื่อEN","ขอบข่ายTH","ขอบข่ายEN",
              "วันที่ประกาศ","หมวดหมู่","สถานะ","หน่วยงาน","บังคับ"]

# Combine Thai + English scope for richer features
df["text"] = (df["ขอบข่ายEN"].fillna("") + " " + df["ชื่อEN"].fillna("")).str.strip()
df = df[df["text"].str.len() > 10].copy()  # remove empty

# Filter out categories with < 2 samples (needed for stratified split)
cat_counts = df["หมวดหมู่"].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
df = df[df["หมวดหมู่"].isin(valid_cats)].copy()

print(f"Records with text: {len(df)}")
print(f"Categories (≥2 samples): {df["หมวดหมู่"].nunique()}")
print(df["หมวดหมู่"].value_counts())

## TF-IDF Vectorization

In [ ]:
# Use English scope text for TF-IDF (cleaner tokenization)
tfidf = TfidfVectorizer(max_features=500, stop_words='english',
                        ngram_range=(1,2), min_df=2)
X_tfidf = tfidf.fit_transform(df['text'])

le = LabelEncoder()
y = le.fit_transform(df['หมวดหมู่'])

print(f'TF-IDF matrix: {X_tfidf.shape}')
print(f'Classes: {len(le.classes_)}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.25, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Model 1 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=15,
                             random_state=42, n_jobs=-1,
                             class_weight='balanced')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'🎯 Random Forest Accuracy: {acc_rf:.4f} ({acc_rf*100:.1f}%)')
print(f'\n📋 Classification Report:\n')
print(classification_report(y_test, y_pred_rf, 
                            zero_division=0))

## Model 2 — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42,
                         class_weight='balanced', C=1.0)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'🎯 Logistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%)')
print(f'\n📋 Classification Report:\n')
print(classification_report(y_test, y_pred_lr, 
                            zero_division=0))

## Cross-Validation Comparison

In [ ]:
cv_rf = cross_val_score(rf, X_tfidf, y, cv=5, scoring='accuracy')
cv_lr = cross_val_score(lr, X_tfidf, y, cv=5, scoring='accuracy')

print(f'5-Fold CV:')
print(f'  Random Forest: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
print(f'  Logistic Reg:  {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')

fig_cv = go.Figure()
fig_cv.add_trace(go.Box(y=cv_rf, name='Random Forest',
    marker_color='#003049', boxpoints='all'))
fig_cv.add_trace(go.Box(y=cv_lr, name='Logistic Regression',
    marker_color='#d62828', boxpoints='all'))
fig_cv.update_layout(
    title='5-Fold CV Accuracy Comparison',
    yaxis_title='Accuracy', template='plotly_white', height=400,
    font=dict(family='Sarabun, sans-serif'),
)
fig_cv.show()

## Confusion Matrix — Best Model

In [ ]:
# Use the better model
best_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
best_pred = y_pred_rf if acc_rf >= acc_lr else y_pred_lr
best_acc = max(acc_rf, acc_lr)

cm = confusion_matrix(y_test, best_pred)
# Filter to classes that appear in test set
present = sorted(set(y_test) | set(best_pred))
present_names = [le.classes_[i] for i in present]

fig_cm = px.imshow(
    cm, text_auto=True,
    x=present_names, y=present_names,
    labels={'x':'Predicted','y':'Actual','color':'Count'},
    color_continuous_scale='YlOrRd',
    title=f'Confusion Matrix — {best_name} (Acc: {best_acc:.1%})',
)
fig_cm.update_layout(
    font=dict(family='Sarabun, sans-serif'),
    height=550, width=650,
)
fig_cm.show()

## Top TF-IDF Features

In [ ]:
feature_names = tfidf.get_feature_names_out()
# Get top features from RF
importances = rf.feature_importances_
top_idx = np.argsort(importances)[-20:]

fig_feat = go.Figure(go.Bar(
    x=importances[top_idx],
    y=[feature_names[i] for i in top_idx],
    orientation='h',
    marker=dict(color=importances[top_idx],
                colorscale=[[0,'#fcbf49'],[1,'#d62828']]),
    text=[f'{v:.4f}' for v in importances[top_idx]],
    textposition='outside',
))
fig_feat.update_layout(
    title='Top 20 TF-IDF Features (Random Forest Importance)',
    xaxis_title='Importance', template='plotly_white',
    height=520, font=dict(family='Sarabun, sans-serif'),
    margin=dict(l=200),
)
fig_feat.show()

## สรุป

Text Classification ใช้ TF-IDF จากขอบข่ายภาษาอังกฤษ + ชื่อมาตรฐาน เพื่อทำนายหมวดหมู่ มอก.

- เปรียบเทียบ Random Forest vs Logistic Regression
- ใช้ 5-Fold Cross Validation เพื่อประเมินความเสถียร
- Feature สำคัญ เช่น คำเฉพาะของหมวดหมู่ (steel, cable, voltage ฯลฯ) มี importance สูง